# NHANES Diabetes Model Comparison

This notebook loads the analysis-ready NHANES dataset, trains four classifiers, and lets you inspect the prediction scores for one selected model at a time.

In [26]:
import importlib.util
import subprocess
import sys

required_packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
    "xgboost": "xgboost",
    "ipywidgets": "ipywidgets",
    "jinja2": "jinja2",
}

missing_packages = [
    package_name
    for module_name, package_name in required_packages.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])

import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
widgets.__version__

'8.1.8'

## Import Libraries and Load the Dataset

Load the analysis-ready NHANES dataset that already contains the diabetes outcome and cleaned features.

In [27]:
from pathlib import Path

data_path = Path(r"c:\Users\jayde\OneDrive - Hogeschool Rotterdam\Vakken\AI in Healthcare\G2_Feature_Importance_DT2_Prediction\DATASETS\nhanes_diabetes_analysis_ready.csv")
df = pd.read_csv(data_path)

print(f"Loaded dataset: {data_path}")
print(f"Shape: {df.shape}")
display(df.head())

Loaded dataset: c:\Users\jayde\OneDrive - Hogeschool Rotterdam\Vakken\AI in Healthcare\G2_Feature_Importance_DT2_Prediction\DATASETS\nhanes_diabetes_analysis_ready.csv
Shape: (26001, 27)


,participant_id,cycle,diabetes,age,sex,race_ethnicity,education_level,income_poverty_ratio,bmi,waist_circumference,mean_systolic_bp,mean_diastolic_bp,ever_smoked_100_cigarettes,average_alcoholic_drinks_per_day,vigorous_work_activity,moderate_work_activity,walk_or_bicycle_transport,vigorous_recreational_activity,moderate_recreational_activity,sleep_hours,energy_kcal,protein_g,carbohydrate_g,total_sugar_g,fiber_g,total_fat_g,cholesterol_mg
0,62161.0,2011-2012,0,22.0,1.0,3.0,3.0,3.15,23.3,81.0,110.666667,74.666667,2.0,NaN,2.0,2.0,2.0,2.0,2.0,8.0,2969.0,104.68,359.59,109.09,18.6,123.81,328.0
1,62164.0,2011-2012,0,44.0,2.0,3.0,4.0,1.67,23.2,80.1,118.000000,60.000000,2.0,NaN,1.0,2.0,2.0,1.0,1.0,8.0,1115.0,73.13,91.67,32.29,9.5,51.54,207.0
2,62169.0,2011-2012,0,21.0,1.0,6.0,3.0,0.33,20.1,69.6,124.666667,78.000000,2.0,2.0,2.0,2.0,2.0,2.0,2.0,8.0,1831.0,77.46,297.51,78.19,4.3,34.61,205.0
3,62172.0,2011-2012,0,43.0,2.0,4.0,3.0,2.02,33.3,120.4,102.000000,71.333333,1.0,3.0,2.0,2.0,2.0,2.0,2.0,8.0,1845.0,57.43,192.82,58.56,2.8,42.02,160.0
4,62174.0,2011-2012,0,80.0,1.0,3.0,5.0,4.30,33.9,116.5,97.000000,39.000000,2.0,NaN,2.0,2.0,2.0,2.0,2.0,9.0,1178.0,39.35,146.72,84.63,12.0,49.75,468.0


## Prepare Features and Target

Separate the diabetes label from the feature columns and prepare a consistent train/test split.

In [28]:
target_column = "diabetes"
feature_columns = [column for column in df.columns if column not in {"participant_id", "cycle", target_column}]

X = df[feature_columns].copy()
y = df[target_column].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

print(f"Feature columns: {len(feature_columns)}")
print(f"Training rows: {X_train.shape[0]}")
print(f"Testing rows: {X_test.shape[0]}")
print("Target balance:")
display(y.value_counts(normalize=True).rename("share").to_frame())

Feature columns: 24
Training rows: 20800
Testing rows: 5201
Target balance:


,share
diabetes,
0,0.8112
1,0.1888


## Split Data for Training and Testing

Keep one shared split so every model is evaluated on the same test set.

## Import and Define the Four Models

Create one pipeline per model so each classifier can be trained consistently on the same feature split.

In [29]:
models = {
    "Logistic Regression": Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            (
                "model",
                LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42),
            ),
        ]
    ),
    "Decision Tree": Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("model", DecisionTreeClassifier(class_weight="balanced", random_state=42)),
        ]
    ),
    "Random Forest": Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=300,
                    class_weight="balanced",
                    random_state=42,
                    n_jobs=-1,
                ),
            ),
        ]
    ),
    "XGBoost": Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            (
                "model",
                XGBClassifier(
                    n_estimators=300,
                    learning_rate=0.05,
                    max_depth=3,
                    subsample=0.9,
                    colsample_bytree=0.9,
                    eval_metric="logloss",
                    random_state=42,
                    n_jobs=-1,
                ),
            ),
        ]
    ),
}

model_results = {}
for model_name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    predicted_labels = pipeline.predict(X_test)
    predicted_probabilities = pipeline.predict_proba(X_test)[:, 1]
    model_results[model_name] = {
        "model": pipeline,
        "accuracy": accuracy_score(y_test, predicted_labels),
        "precision": precision_score(y_test, predicted_labels, zero_division=0),
        "recall": recall_score(y_test, predicted_labels, zero_division=0),
        "f1": f1_score(y_test, predicted_labels, zero_division=0),
        "roc_auc": roc_auc_score(y_test, predicted_probabilities),
        "predictions": predicted_labels,
        "probabilities": predicted_probabilities,
    }

summary_table = pd.DataFrame(
    {
        name: {
            "accuracy": result["accuracy"],
            "precision": result["precision"],
            "recall": result["recall"],
            "f1": result["f1"],
            "roc_auc": result["roc_auc"],
        }
        for name, result in model_results.items()
    }
).T

summary_table.sort_values("roc_auc", ascending=False)

,accuracy,precision,recall,f1,roc_auc
XGBoost,0.814651,0.525714,0.187373,0.276276,0.799097
Random Forest,0.815420,0.555000,0.113035,0.187817,0.791535
Logistic Regression,0.694097,0.350369,0.726069,0.472655,0.784139
Decision Tree,0.755624,0.353597,0.355397,0.354495,0.602172


## Train Each Model

Fit all four pipelines on the training set so their prediction scores can be compared fairly.

## Evaluate Model Prediction Scores

Summarize the predictions on the test split and keep the results ready for interactive inspection.

In [30]:
display(summary_table.round(3).sort_values("roc_auc", ascending=False))

,accuracy,precision,recall,f1,roc_auc
XGBoost,0.815,0.526,0.187,0.276,0.799
Random Forest,0.815,0.555,0.113,0.188,0.792
Logistic Regression,0.694,0.350,0.726,0.473,0.784
Decision Tree,0.756,0.354,0.355,0.354,0.602


## Add Interactive Model Selection

Use a dropdown widget to choose one of the four trained models and inspect its scores and example predictions.

In [31]:
selected_model_dropdown = widgets.Dropdown(
    options=list(models.keys()),
    value="Logistic Regression",
    description="Model:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="320px"),
)

output_area = widgets.Output()


def display_selected_model_scores(model_name: str) -> None:
    result = model_results[model_name]
    selected_summary = pd.DataFrame(
        {
            "metric": ["accuracy", "precision", "recall", "f1", "roc_auc"],
            "score": [
                result["accuracy"],
                result["precision"],
                result["recall"],
                result["f1"],
                result["roc_auc"],
            ],
        }
    )
    selected_predictions = pd.DataFrame(
        {
            "actual": y_test.reset_index(drop=True).head(10),
            "predicted": pd.Series(result["predictions"]).head(10),
            "predicted_probability": pd.Series(result["probabilities"]).head(10),
        }
    )

    with output_area:
        clear_output(wait=True)
        print(f"Selected model: {model_name}")
        display(selected_summary.round(3))
        display(selected_predictions.round(3))


def on_model_change(change):
    if change["name"] == "value" and change["new"] is not None:
        display_selected_model_scores(change["new"])

selected_model_dropdown.observe(on_model_change)
display(selected_model_dropdown, output_area)
display_selected_model_scores(selected_model_dropdown.value)

Dropdown(description='Model:', layout=Layout(width='320px'), options=('Logistic Regression', 'Decision Tree', …

Output()

## Display Selected Model Score and Predictions

The widget below updates the score table and the first ten test-set predictions for the selected model.